# 10) Decision Agent (A/B Chooser)

This notebook builds a very small, engaging Python "agent" using the GAME structure:

- G — Goals: what success looks like and simple rules
- A — Actions: the tools the agent can call (plain Python functions)
- M — Memory: small persistent state (saved to a JSON file)
- E — Environment: executes tools and returns feedback to the agent loop

Learning objective: Students understand the agent loop (decide -> act -> observe -> repeat) with minimal coding.

Tip for class demos: If the agent uses waiting/timers, enable DEMO_MODE=True to shorten waits.


## G — Goals

Help a student choose between Option A and Option B using weighted criteria (time, difficulty, impact).

## Simple rules

- Ask options.
- Ask weights (1-3).
- Ask scores (1-5).
- Recommend and save history.

## How to run

1. Run all cells top-to-bottom.
2. When the agent loop starts, interact in the notebook input prompts.
3. Stop anytime by typing: `stop` (or `exit`, `quit`).


In [ ]:
from __future__ import annotations

import json, os, random, time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

def should_stop(text: str) -> bool:
    return text.strip().lower() in {"stop", "exit", "quit", "end"}

@dataclass
class Action:
    # Structured action output: tool_name + args
    tool_name: str
    args: Dict[str, Any]

class Tools:
    # A: Actions (Tools) - tiny functions the agent can call
    @staticmethod
    def notify(message: str) -> str:
        print(f"\n[Agent] {message}")
        return "ok"

    @staticmethod
    def ask(prompt: str) -> str:
        return input(f"\n[You] {prompt}\n> ").strip()

    @staticmethod
    def load_memory(path: str) -> Dict[str, Any]:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        return {}

    @staticmethod
    def save_memory(path: str, data: Dict[str, Any]) -> str:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        return "saved"

class Environment:
    # E: Environment - runs tools and returns feedback + updated memory
    def __init__(self, memory_path: str):
        self.memory_path = memory_path

    def execute(self, action: Action, memory: Dict[str, Any]) -> Dict[str, Any]:
        if action.tool_name == "notify":
            Tools.notify(str(action.args.get("message", "")))
            return {"result": "notified", "memory": memory}

        if action.tool_name == "ask":
            ans = Tools.ask(str(action.args.get("prompt", "")))
            return {"result": ans, "memory": memory}

        if action.tool_name == "load_memory":
            loaded = Tools.load_memory(self.memory_path)
            return {"result": "loaded", "memory": loaded}

        if action.tool_name == "save_memory":
            Tools.save_memory(self.memory_path, memory)
            return {"result": "saved", "memory": memory}

        if action.tool_name == "terminate":
            return {"result": "terminated", "memory": memory}

        return {"result": "unknown_tool", "memory": memory}


## Implementation (GAME Agent Loop)
The code below is intentionally small and readable.

In [ ]:
MEMORY_FILE = "10_decision_memory.json"
CRITERIA = ["time", "difficulty", "impact"]

def score(scores: Dict[str, int], weights: Dict[str, int]) -> int:
    return sum(int(scores[c]) * int(weights[c]) for c in CRITERIA)

def run_agent():
    env = Environment(MEMORY_FILE)
    memory = env.execute(Action("load_memory", {}), {})["memory"] or {}
    history = memory.get("history", [])

    Tools.notify("Decision Agent is running. Type 'stop' to end.")

    while True:
        a = Tools.ask("Option A:")
        if should_stop(a): break
        b = Tools.ask("Option B:")
        if should_stop(b): break

        Tools.notify("Weights for criteria (1-3):")
        weights = {}
        for c in CRITERIA:
            w = Tools.ask(f"Weight for {c}:")
            weights[c] = int(w) if w.isdigit() else 2

        Tools.notify("Scores for each option (1-5). Higher is better.")
        sa, sb = {}, {}
        for c in CRITERIA:
            x = Tools.ask(f"Score A for {c}:"); sa[c] = int(x) if x.isdigit() else 3
            y = Tools.ask(f"Score B for {c}:"); sb[c] = int(y) if y.isdigit() else 3

        total_a = score(sa, weights)
        total_b = score(sb, weights)

        if total_a == total_b:
            rec = "Tie. Pick the one you can start in 5 minutes."
        elif total_a > total_b:
            rec = f"Recommend A: {a}"
        else:
            rec = f"Recommend B: {b}"

        Tools.notify(f"Score A={total_a}, Score B={total_b}\n{rec}")

        history.append({"A": a, "B": b, "weights": weights, "scoresA": sa, "scoresB": sb,
                        "totalA": total_a, "totalB": total_b, "rec": rec, "ts": time.time()})
        memory["history"] = history[-50:]
        env.execute(Action("save_memory", {}), memory)

        again = Tools.ask("Another decision? (yes/no)")
        if again.lower().startswith("n"):
            break

    env.execute(Action("save_memory", {}), memory)
    Tools.notify("Saved. Bye!")

run_agent()


## Easy extensions

- Add a third option.
- Confidence rating.
- Review last decision.